# CAMEL Playground notebook (SENG 691 Group 3, Phase 1)

[CAMEL](https://github.com/camel-ai/camel) makes two AI agents work on a task together: one plays the person who wants something done and gives instructions, the other plays the expert who does the work. A third agent first sharpens the task. This notebook lets you watch and steer that.

**Two ways to use it**

* **Quick start.** Run the cells down to *Quick start* and a small app opens: type any task, press Start, watch the agents one turn at a time, read the Narrator's plain-English summaries, and step in with your own notes or questions.
* **Walkthrough.** The cells after that run the same engine one step at a time on a fixed task, so you can see each piece: the sharpened task, the rulebooks each agent gets, the turns, and the Narrator.

**Running it**

* *Google Colab:* Runtime > Run all. The setup cell installs what is missing and asks you to upload `demo/playground.py` from the repo once. Put your key in Colab's **Secrets** (key icon) under its usual name, e.g. `OPENROUTER_API_KEY` or `OPENAI_API_KEY`; otherwise you are asked for it in a hidden box.
* *Locally:* set up with `setup.sh` or `setup.ps1`, start Jupyter from the repo, pick the venv kernel, Run all.

**Choosing a model.** Leave the settings alone and you will be asked which provider to use the first time a model is needed (OpenRouter's free models, Groq, Gemini and Cerebras cost nothing; OpenAI and Anthropic are paid; Ollama runs locally). Or preset it in the next cell, or in `.env` with `MODEL_PLATFORM` and the key. Keys are never printed or saved into this notebook.


In [1]:
# ---- Settings (optional). Leave as is to be asked for a provider and key when needed. ----
STUB = False       # True = offline canned replies: no key, no cost, no intelligence (plumbing test only)
PLATFORM = None    # "openrouter", "openai", "groq", "gemini", "cerebras", "anthropic", "ollama"; None = ask / .env
MODEL = None       # model name on that provider; None = the provider's default

# Walkthrough section only (the Playground asks for its own task):
TASK_PROMPT = "Develop a trading bot for the stock market"
ASSISTANT_ROLE, USER_ROLE = "Python Programmer", "Stock Trader"
TURNS = 5          # each turn = 2 model calls
CONCISE = True     # ask both agents for short, plain-language messages (code only on request)

In [2]:
# ---- Setup: run once per session. Installs what is missing and locates our playground module. ----
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

CAMEL_VERSION = "0.2.91a7"  # same version as the camel/ submodule (upstream commit 797296a)

def _missing(pkg):
    spec = importlib.util.find_spec(pkg)
    return spec is None or spec.origin is None   # origin None = empty namespace folder, not an install

if _missing("camel"):
    print(f"Installing camel-ai=={CAMEL_VERSION} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"camel-ai=={CAMEL_VERSION}"])
if _missing("gradio"):
    print("Installing gradio (for the Playground) ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gradio"])

ROOT = Path.cwd() if (Path.cwd() / ".env.example").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "demo"))              # inside the repo: demo/playground.py is here
if importlib.util.find_spec("playground") is None:  # Colab: only the notebook was uploaded
    try:
        from google.colab import files
        print("Please upload demo/playground.py from the repo (file picker below).")
        files.upload()
        sys.path.insert(0, ".")
    except ImportError:
        raise FileNotFoundError("demo/playground.py not found. Run this notebook from the repo, "
                                "or copy playground.py next to it.")

import logging
import warnings
warnings.filterwarnings("ignore")                  # keep library warnings out of the cell outputs
logging.getLogger("camel").setLevel(logging.ERROR)

from dotenv import load_dotenv
from IPython.display import HTML, Markdown, display
import camel
import playground

load_dotenv(ROOT / ".env")
print(f"CAMEL {camel.__version__} ready.")

CAMEL 0.2.91a7 ready.


## Quick start: the Playground

Run the cell below and the app appears under it (locally it also opens in a browser tab).

1. Type **any** task in the one box and press **Start**. A model picks the two roles for you (open *Advanced* to set roles or the provider yourself).
2. Press **Next turn** to watch one exchange at a time, or **Run 3 turns**. Each turn appears as it lands.
3. After each burst the **Narrator** posts in the chat, in plain English: what just happened, how it is going, done or not, and a nudge you could give. The strip above the chat always shows the goal, who is who, the turn count, and whether the task has been declared done.
4. Step in any time with the box at the bottom: a note both agents see before the next turn, or a question for the assistant agent, which the agents then continue from.

If you have not preset a provider, the app's *Advanced* fold shows the provider and asks for nothing else; the key comes from `.env`, a Colab secret, or the Playground's error message tells you which variable to set.

In [ ]:
# Open the Playground.
import gradio as gr

ui = playground.build_ui(
    default_platform=PLATFORM or os.getenv("MODEL_PLATFORM") or "openrouter",
    default_model=MODEL or os.getenv("MODEL_NAME"),
    default_stub=STUB,
)
ui.launch(inline=True, share=False, quiet=True, theme=gr.themes.Soft())

## Walkthrough: the same engine, one step at a time

Think of a client and a contractor on a phone call, with a project manager who first clarifies the brief:

| Step | Who | What happens |
|---|---|---|
| 1 | **Task specifier** (project manager) | Rewrites the vague task into a concrete spec |
| 2 | Framework | Hands each agent a *rulebook* (system message) that locks it into its role |
| 3 | **AI User** (client; here the Stock Trader) | Gives **one instruction per turn**, never asks questions |
| 4 | **AI Assistant** (contractor; here the Python Programmer) | Answers each instruction with a `Solution:` and ends with `Next request.` |
| 5 | Loop | Steps 3 and 4 repeat until the client says `CAMEL_TASK_DONE` or the turn cap is hit |

The cells below use the same `Playground` class as the app, so what you see here is exactly what happens behind the buttons.

### Step 1: choose a model and sharpen the brief

If nothing is preset, the next cell asks for a provider (and a key, in a hidden box). `TASK_PROMPT` is deliberately vague; the task-specifier agent rewrites it before the two main agents start. With `CONCISE = True` a short style rule is added after sharpening, asking both agents for messages under about 120 words with code only on request; set it to `False` to see CAMEL's default, much longer, style.

In [4]:
model, mode = playground.choose_model(PLATFORM, MODEL, STUB)   # asks only for what is not preset
pg = playground.Playground()
pg.start(TASK_PROMPT, ASSISTANT_ROLE, USER_ROLE, PLATFORM, MODEL, STUB,
         specify=True, model=model, mode=mode, concise=CONCISE)
display(Markdown(pg.goal_markdown()))

**Model:** gpt-4.1-mini-2025-04-14 via openai

**Original task:** Develop a trading bot for the stock market

**Sharpened by the task-specifier agent:** Develop a Python-based trading bot that analyzes real-time market data, applies machine learning to predict short-term price movements, and executes automated buy/sell orders on NASDAQ stocks with risk management and customizable trading strategies.

**Roles:** Python Programmer does the work; Stock Trader gives one instruction per turn.

**Turns completed:** 0

**Status:** In progress. Run more turns, or step in below.

### Step 2: the rulebooks

CAMEL writes a system message for each agent from the role names and the sharpened task. They are collapsed below; expand one to see how strictly the roles are enforced (the trader must never ask questions; the programmer must always start with `Solution:`).

In [5]:
from html import escape

def collapsed(title, text):
    return (f"<details><summary><b>{title}</b> (click to expand)</summary>"
            f"<pre style=\"white-space:pre-wrap\">{escape(text)}</pre></details>")

display(HTML(collapsed(f"Rulebook for {pg.assistant_role} (AI Assistant)", pg.session.assistant_sys_msg.content)))
display(HTML(collapsed(f"Rulebook for {pg.user_role} (AI User)", pg.session.user_sys_msg.content)))

### Steps 3 to 5: the conversation, then the Narrator

Each turn has two parts: the **AI User** gives one `Instruction:` (plus an optional `Input:`), the **AI Assistant** replies with `Solution:`, code or text, and `Next request.` Read the turn headings first; they are the instructions in one line. When the turns are done, the Narrator explains the whole run in plain English, exactly as it does after each burst in the Playground.

In [6]:
def first_line(text, prefix="Instruction:"):
    for line in text.splitlines():
        if line.strip().startswith(prefix):
            return line.strip()[len(prefix):].strip()
    return text.strip().splitlines()[0] if text.strip() else ""

run = []
for n in range(1, TURNS + 1):
    new = pg.run_turns(1)
    if not new:
        break
    run += new
    user_msgs = [t for who, t in new if who == pg.user_role]
    display(Markdown(f"---\n### Turn {pg.turn}: {first_line(user_msgs[0]) if user_msgs else ''}"))
    for who, text in new:
        display(Markdown(f"**{who}:**\n\n{text}"))
    if pg.done or pg.terminated:
        break

display(Markdown("---\n## Narrator\n\n" + pg.narrate(run)))
display(Markdown(pg.goal_markdown()))

---
### Turn 1: Set up a Python environment with necessary packages for data analysis and trading, including pandas, numpy, scikit-learn, and a NASDAQ market data API like Alpha Vantage or yFinance.

**Stock Trader:**

Instruction: Set up a Python environment with necessary packages for data analysis and trading, including pandas, numpy, scikit-learn, and a NASDAQ market data API like Alpha Vantage or yFinance.
Input: None

**Python Programmer:**

Solution:  
Set up a Python environment using virtualenv or conda for isolation. Then, install needed packages via pip:

1. pandas (data manipulation)  
2. numpy (numerical ops)  
3. scikit-learn (ML models)  
4. yfinance (for NASDAQ stock data; simpler than Alpha Vantage)  

Commands:  
```bash  
pip install pandas numpy scikit-learn yfinance  
```  

This setup covers data handling, ML, and market data access for the bot. Next request.

---
### Turn 2: Write Python code to fetch real-time NASDAQ stock data using yfinance for a chosen symbol (e.g., AAPL), retrieving the last 1-minute interval data.

**Stock Trader:**

Instruction: Write Python code to fetch real-time NASDAQ stock data using yfinance for a chosen symbol (e.g., AAPL), retrieving the last 1-minute interval data.
Input: None

...

---
### Turn 5: Split the dataset into training and testing sets (e.g., 80% train, 20% test) for model training and evaluation.

**Stock Trader:**

Instruction: Split the dataset into training and testing sets (e.g., 80% train, 20% test) for model training and evaluation.
Input: Use `df` with features and target. Separate features (return, MA_5, MA_15, vol_change) and target columns.

**Python Programmer:**

Solution:  
Split `df` into train/test by 80/20 without shuffling (time series). Features: return, MA_5, MA_15, vol_change; target: target.

Code:

```python
from sklearn.model_selection import train_test_split

features = ['return', 'MA_5', 'MA_15', 'vol_change']
X = df[features]
y = df['target']

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
```

This keeps chronological order for training and testing. Next request.

---
## Narrator

Just now: The programmer split the prepared data into training and testing sets preserving time order for model building.  
Going well? Yes, they are following the goal without repeating or stalling.  
Done? Partly, because model training and evaluation still needs to happen.  
Nudge: Ask them to train a machine learning model and evaluate its accuracy next.

**Model:** gpt-4.1-mini-2025-04-14 via openai

**Original task:** Develop a trading bot for the stock market

**Sharpened by the task-specifier agent:** Develop a Python-based trading bot that analyzes real-time market data, applies machine learning to predict short-term price movements, and executes automated buy/sell orders on NASDAQ stocks with risk management and customizable trading strategies.

**Roles:** Python Programmer does the work; Stock Trader gives one instruction per turn.

**Turns completed:** 5

**Status:** In progress. Run more turns, or step in below.

## What this demonstrates

* **Task specification**: an agent turned a one-line brief into a concrete spec (Step 1).
* **Role-playing with inception prompting**: two agents stayed in character because of the rulebooks (Step 2).
* **Autonomous multi-turn cooperation**: the agents built on each other's messages without human input (Steps 3 to 5).
* **Human in the loop** (Playground): guidance and questions between bursts change what the agents do next.

## Notes

* `TURNS` caps the walkthrough; the upstream example runs up to 50 turns, which costs real money. Hitting the cap before `CAMEL_TASK_DONE` is expected, not an error.
* Smaller free models follow the `Instruction:` / `Solution:` format less strictly than paid ones; the Narrator will say so when the agents drift.
* The upstream CAMEL source in `camel/` is unmodified. Our code is `demo/playground.py`, `demo/role_playing_demo.py` and this notebook.